In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from arch import arch_model
import plotly.graph_objects as go
from plotly.subplots import make_subplots

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:

class VolatilityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class GatingDataset(Dataset):
    def __init__(self, X, y_real, y_garch):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y_real = torch.FloatTensor(y_real)
        self.y_garch = torch.FloatTensor(y_garch)
    def __len__(self): return len(self.y_real)
    def __getitem__(self, idx): return self.X[idx], self.y_real[idx], self.y_garch[idx]

class VolatilityLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=256, num_layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.batch_norm = nn.BatchNorm1d(hidden_size)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        normalized = self.batch_norm(last_output)
        fc1_out = self.relu(self.fc1(normalized))
        fc1_out = self.dropout(fc1_out)
        return self.fc2(fc1_out).squeeze(-1)

class FeatureGatingGINN(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.data_lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.data_bn = nn.BatchNorm1d(hidden_size)
        self.garch_proj = nn.Sequential(
            nn.Linear(1, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, hidden_size)
        )
        self.lambda_lstm = nn.LSTM(input_size, 32, 2, batch_first=True, bidirectional=True)
        self.lambda_bn = nn.BatchNorm1d(64)
        self.lambda_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1)
        )
    def forward(self, x, garch_signal):
        out, _ = self.data_lstm(x)
        data_feat = self.data_bn(out[:, -1, :])
        garch_feat = self.garch_proj(garch_signal)
        l_out, _ = self.lambda_lstm(x)
        l_feat = self.lambda_bn(l_out[:, -1, :])
        lambda_val = self.lambda_head(l_feat)
        combined = data_feat + lambda_val * garch_feat
        output = self.output_head(combined).squeeze(-1)
        return output, lambda_val.squeeze(-1)

In [ ]:
def prepare_data(prices_or_df, returns_or_dates=None, split_dates=None, index_name='INDEX'):
    """
    Универсальная подготовка данных.
    - Для IMOEX: передать returns_imoex (pd.Series), остальное None
    - Для NASDAQ: передать (prices_array, dates_array)
    """
    if returns_or_dates is not None:
        # Режим NASDAQ
        prices = np.log(prices_or_df)
        returns = pd.Series(np.diff(prices) * 100)
        all_dates = returns_or_dates[1:]  # dates для returns (на 1 короче из-за diff)
    else:
        # Режим IMOEX
        returns = prices_or_df.copy()
        all_dates = imoex_df['TRADEDATE'].values[1:]  # dates для returns

    print(f'[{index_name}] Доходностей: {len(returns)}')

    # Ground Truth Variance
    rolling_mean = returns.rolling(window=90).mean()
    raw_var = (returns - rolling_mean) ** 2
    ground_truth_var = raw_var.rolling(window=5).mean()

    # Берём не-dropna значения
    gt_var = ground_truth_var.dropna().values
    gt_var_log = np.log1p(gt_var)
    
    # Индексы в исходном returns, соответствующие gt_var
    gt_indices = ground_truth_var.dropna().index.values
    
    # ИСПРАВЛЕНИЕ: Ограничиваем индексы длиной all_dates
    max_idx = len(all_dates) - 1
    valid_mask = gt_indices <= max_idx
    gt_indices = gt_indices[valid_mask]
    gt_var = gt_var[valid_mask]
    gt_var_log = gt_var_log[valid_mask]
    
    # Даты для gt_var
    gt_dates = all_dates[gt_indices]

    print(f'[{index_name}] GT variance точек: {len(gt_var)}')

    # Формирование окон
    WINDOW = 90
    X_windows, y_targets, window_dates = [], [], []
    
    # i пробегает по gt_var_log, начиная с WINDOW
    for i in range(WINDOW, len(gt_var_log)):
        X_windows.append(gt_var_log[i - WINDOW : i])
        y_targets.append(gt_var_log[i])
        window_dates.append(gt_dates[i])  # теперь длины совпадают

    X_windows = np.array(X_windows)
    y_targets = np.array(y_targets)
    window_dates = np.array(window_dates)
    print(f'[{index_name}] Окна: {X_windows.shape}')

    # Разбиение на train/val/test
    window_dates_dt = window_dates.astype('datetime64')
    if split_dates is None:
        # IMOEX
        train_mask = window_dates_dt < np.datetime64('2020-01-01')
        val_mask = (window_dates_dt >= np.datetime64('2020-01-01')) & (window_dates_dt < np.datetime64('2022-01-01'))
        test_mask = window_dates_dt >= np.datetime64('2022-01-01')
    else:
        # NASDAQ
        train_mask = window_dates_dt < np.datetime64(split_dates['train_end'])
        val_mask = (window_dates_dt >= np.datetime64(split_dates['val_start'])) & (window_dates_dt < np.datetime64(split_dates['val_end']))
        test_mask = window_dates_dt >= np.datetime64(split_dates['test_start'])

    X_train, y_train = X_windows[train_mask], y_targets[train_mask]
    X_val, y_val = X_windows[val_mask], y_targets[val_mask]
    X_test, y_test = X_windows[test_mask], y_targets[test_mask]
    dates_test = window_dates[test_mask]

    print(f'[{index_name}] Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

    # Нормализация
    train_mean = X_train.mean()
    train_std = X_train.std()
    
    X_train_norm = (X_train - train_mean) / train_std
    X_val_norm = (X_val - train_mean) / train_std
    X_test_norm = (X_test - train_mean) / train_std
    y_train_norm = (y_train - train_mean) / train_std
    y_val_norm = (y_val - train_mean) / train_std
    y_test_norm = (y_test - train_mean) / train_std

    # GARCH rolling forecast
    print(f'[{index_name}] Считаю GARCH для всех окон...')
    garch_all_preds = []
    for i in range(len(X_windows)):
        # Позиция в исходном returns
        pos = gt_indices[WINDOW + i]
        train_data = returns.iloc[pos - 90 : pos]
        try:
            m = arch_model(train_data, vol='Garch', p=1, q=1, dist='Normal')
            r = m.fit(disp='off')
            f = r.forecast(horizon=1)
            garch_all_preds.append(f.variance.values[-1, 0])
        except:
            garch_all_preds.append(np.nan)
        if (i + 1) % 500 == 0:
            print(f'  {i+1}/{len(X_windows)}')

    garch_all_preds = np.array(garch_all_preds)
    nan_mask = np.isnan(garch_all_preds)
    if nan_mask.sum() > 0:
        garch_all_preds[nan_mask] = np.nanmean(garch_all_preds)
        print(f'  Заполнено {nan_mask.sum()} NaN')

    garch_log = np.log1p(garch_all_preds)
    garch_all_norm = (garch_log - train_mean) / train_std

    garch_train_norm = garch_all_norm[train_mask]
    garch_val_norm = garch_all_norm[val_mask]
    garch_test_norm = garch_all_norm[test_mask]
    garch_test_raw = garch_all_preds[test_mask]

    # Правильный таргет для теста (в оригинальном масштабе)
    targets_real = np.expm1(y_test * train_std + train_mean)

    data_pack = {
        'X_train_norm': X_train_norm, 'X_val_norm': X_val_norm, 'X_test_norm': X_test_norm,
        'y_train_norm': y_train_norm, 'y_val_norm': y_val_norm, 'y_test_norm': y_test_norm,
        'garch_train_norm': garch_train_norm, 'garch_val_norm': garch_val_norm, 'garch_test_norm': garch_test_norm,
        'garch_test_raw': garch_test_raw, 'targets_real': targets_real,
        'train_mean': train_mean, 'train_std': train_std, 'dates_test': dates_test
    }
    return data_pack

In [4]:

df = pd.read_json("./data.json")
df.columns = ["BOARDID", "SECID", "TRADEDATE", "SHORTNAME", "NAME", "CLOSE", "OPEN", "HIGH", "LOW", "VALUE", "DURATION", "YIELD", "DECIMALS", "CAPITALIZATION", "CURRENCYID", "DIVISOR", "TRADINGSESSION", "VOLUME", "TRADE_SESSION_DATE", "RECALC_DATE"]
df['LogReturn'] = np.log(df['CLOSE'] / df['CLOSE'].shift(1))
returns_imoex = df['LogReturn'].dropna() * 100

imoex_data = prepare_data(returns_imoex, None, None, 'IMOEX')

[IMOEX] Доходностей: 4076
[IMOEX] GT variance точек: 3982
[IMOEX] Окна: (3892, 90)
[IMOEX] Train: 2325, Val: 505, Test: 1062
[IMOEX] Считаю GARCH для всех окон...
  500/3892
  1000/3892
  1500/3892
  2000/3892
  2500/3892
  3000/3892
  3500/3892


In [9]:
# Обучение моделей на IMOEX
results_imoex = {}

for model_type in ['LSTM', 'GINN_adaptive']:
    print(f'\n=== IMOEX: Обучение {model_type} ===')
    
    # DataLoaders
    train_ld = DataLoader(VolatilityDataset(imoex_data['X_train_norm'], imoex_data['y_train_norm']), 128, shuffle=True)
    val_ld = DataLoader(VolatilityDataset(imoex_data['X_val_norm'], imoex_data['y_val_norm']), 128)
    test_ld = DataLoader(VolatilityDataset(imoex_data['X_test_norm'], imoex_data['y_test_norm']), 128)

    ginn_train_ld = DataLoader(GatingDataset(imoex_data['X_train_norm'], imoex_data['y_train_norm'], imoex_data['garch_train_norm']), 256, shuffle=True)
    ginn_val_ld = DataLoader(GatingDataset(imoex_data['X_val_norm'], imoex_data['y_val_norm'], imoex_data['garch_val_norm']), 128)
    ginn_test_ld = DataLoader(GatingDataset(imoex_data['X_test_norm'], imoex_data['y_test_norm'], imoex_data['garch_test_norm']), 128)

    if model_type == 'LSTM':
        model = VolatilityLSTM(hidden_size=128).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
        criterion = nn.MSELoss()
        
        best_val = float('inf')
        patience_cnt, PATIENCE = 0, 100
        for epoch in range(300):
            model.train()
            for X_b, y_b in train_ld:
                optimizer.zero_grad()
                loss = criterion(model(X_b.to(device)), y_b.to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_b, y_b in val_ld:
                    val_losses.append(criterion(model(X_b.to(device)), y_b.to(device)).item())
            avg_val = np.mean(val_losses)
            scheduler.step(avg_val)
            if avg_val < best_val:
                best_val = avg_val
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'LSTM early stop at epoch {epoch+1}')
                break
            if (epoch+1) % 20 == 0:
                print(f'LSTM Epoch {epoch+1}: Val Loss {avg_val:.4f}')
        model.load_state_dict(best_state)
        
        # Тест
        model.eval()
        preds = []
        with torch.no_grad():
            for X_b, y_b in test_ld:
                preds.extend(model(X_b.to(device)).cpu().numpy())
        preds = np.expm1(np.array(preds) * imoex_data['train_std'] + imoex_data['train_mean'])
        results_imoex['LSTM'] = {'preds': preds}
        
    elif model_type == 'GINN_adaptive':
        model = FeatureGatingGINN(hidden_size=128).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
        mse = nn.MSELoss()
        
        best_val = float('inf')
        patience_cnt = 0
        PATIENCE = 50  # ← ИСПРАВЛЕНО: patience_cnt=0, PATIENCE=50
        for epoch in range(300):
            model.train()
            tr_loss = []
            for X, y_r, y_g in ginn_train_ld:
                X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
                optimizer.zero_grad()
                pred, lam = model(X, y_g.unsqueeze(-1))
                loss = mse(pred, y_r) + 0.001 * ((lam - 0.5)**2).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                tr_loss.append(loss.item())
            
            model.eval()
            val_loss = []
            with torch.no_grad():
                for X, y_r, y_g in ginn_val_ld:
                    X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
                    pred, lam = model(X, y_g.unsqueeze(-1))
                    val_loss.append(mse(pred, y_r).item())
            
            avg_val = np.mean(val_loss)
            scheduler.step(avg_val)
            
            if avg_val < best_val:
                best_val = avg_val
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
            
            if patience_cnt >= PATIENCE:
                print(f'Adaptive GINN early stop at epoch {epoch+1}')
                break
            
            if (epoch+1) % 20 == 0:
                print(f'Adaptive GINN Ep {epoch+1}: Val Loss {avg_val:.4f}')
        
        model.load_state_dict(best_state)

        model.eval()
        preds, lambdas = [], []
        with torch.no_grad():
            for X, y_r, y_g in ginn_test_ld:
                X, y_g = X.to(device), y_g.to(device)
                p, l = model(X, y_g.unsqueeze(-1))
                preds.extend(p.cpu().numpy())
                lambdas.extend(l.cpu().numpy())
        preds = np.expm1(np.array(preds) * imoex_data['train_std'] + imoex_data['train_mean'])
        results_imoex['GINN'] = {'preds': preds, 'lambdas': np.array(lambdas)}


=== IMOEX: Обучение LSTM ===
LSTM Epoch 20: Val Loss 0.1985
LSTM Epoch 40: Val Loss 0.2747
LSTM Epoch 60: Val Loss 0.3145
LSTM Epoch 80: Val Loss 0.1919
LSTM Epoch 100: Val Loss 0.2423
LSTM Epoch 120: Val Loss 0.1883
LSTM Epoch 140: Val Loss 0.1644
LSTM Epoch 160: Val Loss 0.1823
LSTM Epoch 180: Val Loss 0.1676
LSTM Epoch 200: Val Loss 0.1720
LSTM early stop at epoch 212

=== IMOEX: Обучение GINN_adaptive ===
Adaptive GINN Ep 20: Val Loss 0.2051
Adaptive GINN Ep 40: Val Loss 0.1470
Adaptive GINN Ep 60: Val Loss 0.3005
Adaptive GINN Ep 80: Val Loss 0.2310
Adaptive GINN early stop at epoch 86


In [10]:
# Загрузка и подготовка NASDAQ
df_nasdaq = pd.read_csv('dataset\\nasdq.csv')
df_nasdaq['Date'] = pd.to_datetime(df_nasdaq['Date'])
df_nasdaq = df_nasdaq.sort_values('Date').reset_index(drop=True)

nasdaq_data = prepare_data(
    df_nasdaq['Close'].values, 
    df_nasdaq['Date'].values,
    {'train_end': '2018-01-01', 'val_start': '2018-01-01', 'val_end': '2020-01-01', 'test_start': '2020-01-01'},
    'NASDAQ'
)

[NASDAQ] Доходностей: 3913
[NASDAQ] GT variance точек: 3820
[NASDAQ] Окна: (3730, 90)
[NASDAQ] Train: 1928, Val: 529, Test: 1273
[NASDAQ] Считаю GARCH для всех окон...
  500/3730
  1000/3730
  1500/3730
  2000/3730
  2500/3730
  3000/3730
  3500/3730


In [11]:
# Обучение моделей на NASDAQ (аналогично IMOEX)
results_nasdaq = {}

for model_type in ['LSTM', 'GINN_adaptive']:
    print(f'\n=== NASDAQ: Обучение {model_type} ===')
    
    train_ld = DataLoader(VolatilityDataset(nasdaq_data['X_train_norm'], nasdaq_data['y_train_norm']), 128, shuffle=True)
    val_ld = DataLoader(VolatilityDataset(nasdaq_data['X_val_norm'], nasdaq_data['y_val_norm']), 128)
    test_ld = DataLoader(VolatilityDataset(nasdaq_data['X_test_norm'], nasdaq_data['y_test_norm']), 128)

    ginn_train_ld = DataLoader(GatingDataset(nasdaq_data['X_train_norm'], nasdaq_data['y_train_norm'], nasdaq_data['garch_train_norm']), 256, shuffle=True)
    ginn_val_ld = DataLoader(GatingDataset(nasdaq_data['X_val_norm'], nasdaq_data['y_val_norm'], nasdaq_data['garch_val_norm']), 128)
    ginn_test_ld = DataLoader(GatingDataset(nasdaq_data['X_test_norm'], nasdaq_data['y_test_norm'], nasdaq_data['garch_test_norm']), 128)

    if model_type == 'LSTM':
        model = VolatilityLSTM(hidden_size=256).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
        criterion = nn.MSELoss()
        
        best_val, patience_cnt, PATIENCE = float('inf'), 0, 30
        for epoch in range(300):
            model.train()
            for X_b, y_b in train_ld:
                optimizer.zero_grad()
                loss = criterion(model(X_b.to(device)), y_b.to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_b, y_b in val_ld:
                    val_losses.append(criterion(model(X_b.to(device)), y_b.to(device)).item())
            avg_val = np.mean(val_losses)
            scheduler.step(avg_val)
            if avg_val < best_val:
                best_val = avg_val
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'LSTM early stop at epoch {epoch+1}')
                break
            if (epoch+1) % 20 == 0:
                print(f'LSTM Epoch {epoch+1}: Val Loss {avg_val:.4f}')
        model.load_state_dict(best_state)
        
        model.eval()
        preds = []
        with torch.no_grad():
            for X_b, y_b in test_ld:
                preds.extend(model(X_b.to(device)).cpu().numpy())
        preds = np.expm1(np.array(preds) * nasdaq_data['train_std'] + nasdaq_data['train_mean'])
        results_nasdaq['LSTM'] = {'preds': preds}
        
    elif model_type == 'GINN_adaptive':
        model = FeatureGatingGINN(hidden_size=256).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
        mse = nn.MSELoss()
        
        best_val, patience_cnt, PATIENCE = float('inf'), 0, 50
        for epoch in range(300):
            model.train()
            for X, y_r, y_g in ginn_train_ld:
                X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
                optimizer.zero_grad()
                pred, lam = model(X, y_g.unsqueeze(-1))
                loss = mse(pred, y_r) + 0.001 * ((lam - 0.5)**2).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            model.eval()
            val_loss = []
            with torch.no_grad():
                for X, y_r, y_g in ginn_val_ld:
                    X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
                    pred, lam = model(X, y_g.unsqueeze(-1))
                    val_loss.append(mse(pred, y_r).item())
            avg_val = np.mean(val_loss)
            scheduler.step(avg_val)
            if avg_val < best_val:
                best_val = avg_val
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'Adaptive GINN early stop at epoch {epoch+1}')
                break
            if (epoch+1) % 20 == 0:
                print(f'Adaptive GINN Ep {epoch+1}: Val Loss {avg_val:.4f}')
        model.load_state_dict(best_state)
        
        model.eval()
        preds, lambdas = [], []
        with torch.no_grad():
            for X, y_r, y_g in ginn_test_ld:
                X, y_g = X.to(device), y_g.to(device)
                p, l = model(X, y_g.unsqueeze(-1))
                preds.extend(p.cpu().numpy())
                lambdas.extend(l.cpu().numpy())
        preds = np.expm1(np.array(preds) * nasdaq_data['train_std'] + nasdaq_data['train_mean'])
        results_nasdaq['GINN'] = {'preds': preds, 'lambdas': np.array(lambdas)}


=== NASDAQ: Обучение LSTM ===
LSTM Epoch 20: Val Loss 0.1204
LSTM Epoch 40: Val Loss 0.3060
LSTM Epoch 60: Val Loss 0.3417
LSTM Epoch 80: Val Loss 0.0892
LSTM Epoch 100: Val Loss 0.0908
LSTM Epoch 120: Val Loss 0.0902
LSTM early stop at epoch 137

=== NASDAQ: Обучение GINN_adaptive ===
Adaptive GINN Ep 20: Val Loss 0.1257
Adaptive GINN Ep 40: Val Loss 0.1285
Adaptive GINN Ep 60: Val Loss 0.1103
Adaptive GINN Ep 80: Val Loss 0.1050
Adaptive GINN Ep 100: Val Loss 0.0972
Adaptive GINN Ep 120: Val Loss 0.1001
Adaptive GINN Ep 140: Val Loss 0.0930
Adaptive GINN Ep 160: Val Loss 0.0949
Adaptive GINN early stop at epoch 178


## Сравнительный анализ

In [12]:
combined_results = []

for market_name, data, res in [('IMOEX', imoex_data, results_imoex), ('NASDAQ', nasdaq_data, results_nasdaq)]:
    targets = data['targets_real']
    
    # GARCH
    mask = ~np.isnan(data['garch_test_raw'])
    garch_metrics = {
        'Model': 'GARCH(1,1)',
        'Market': market_name,
        'MSE': mean_squared_error(targets[mask], data['garch_test_raw'][mask]),
        'MAE': mean_absolute_error(targets[mask], data['garch_test_raw'][mask]),
        'R2': r2_score(targets[mask], data['garch_test_raw'][mask])
    }
    combined_results.append(garch_metrics)
    
    # LSTM и GINN
    for model_name in ['LSTM', 'GINN']:
        preds = res[model_name]['preds']
        mse_val = mean_squared_error(targets, preds)
        mae_val = mean_absolute_error(targets, preds)
        r2_val = r2_score(targets, preds)
        combined_results.append({
            'Model': model_name,
            'Market': market_name,
            'MSE': mse_val,
            'MAE': mae_val,
            'R2': r2_val
        })

df_combined = pd.DataFrame(combined_results)
summary_table = df_combined.pivot_table(index='Model', columns='Market', values=['MSE', 'MAE', 'R2'])
print("\033[1mСВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ПО РЫНКАМ\033[0m")
print(summary_table.to_string(float_format=lambda x: f'{x:.3f}' if abs(x) < 10 else f'{x:.1f}'))

СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ПО РЫНКАМ
             MAE          MSE            R2       
Market     IMOEX NASDAQ IMOEX NASDAQ  IMOEX NASDAQ
Model                                             
GARCH(1,1) 2.959  2.193 831.7   26.7 -117.9 -0.950
GINN       2.789  2.489 612.7   47.3  -86.6 -2.453
LSTM       1.818  2.165 114.4   17.6  -15.4 -0.283


In [13]:

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('IMOEX: Прогнозы', 'IMOEX: λ(t)', 'NASDAQ: Прогнозы', 'NASDAQ: λ(t)'),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

row_col_map = {'IMOEX': (1, 1), 'NASDAQ': (2, 1)}
lam_col_map = {'IMOEX': (1, 2), 'NASDAQ': (2, 2)}

for market, data, res in [('IMOEX', imoex_data, results_imoex), ('NASDAQ', nasdaq_data, results_nasdaq)]:
    r, c = row_col_map[market]
    
    # Прогнозы
    fig.add_trace(go.Scatter(x=data['dates_test'], y=data['targets_real'], name=f'{market} Real', line=dict(color='black', width=1), opacity=0.5), row=r, col=c)
    fig.add_trace(go.Scatter(x=data['dates_test'], y=data['garch_test_raw'], name=f'{market} GARCH', line=dict(color='red', width=1.5)), row=r, col=c)
    fig.add_trace(go.Scatter(x=data['dates_test'], y=res['LSTM']['preds'], name=f'{market} LSTM', line=dict(color='orange', width=1.5)), row=r, col=c)
    fig.add_trace(go.Scatter(x=data['dates_test'], y=res['GINN']['preds'], name=f'{market} GINN', line=dict(color='blue', width=2)), row=r, col=c)
    
    # Лямбда
    lam_r, lam_c = lam_col_map[market]
    fig.add_trace(go.Scatter(x=data['dates_test'], y=res['GINN']['lambdas'], name=f'{market} λ', line=dict(color='purple'), fill='tozeroy', fillcolor='rgba(128,0,128,0.1)'), row=lam_r, col=lam_c)
    fig.add_hline(y=0.5, line_dash='dash', line_color='green', row=lam_r, col=lam_c)

fig.update_layout(title='Feature Gating GINN: Сравнение IMOEX и NASDAQ', height=900)
fig.show()

---

In [22]:
# Подготовка данных для обоих рынков (с повторным чтением, чтобы не зависеть от предыдущих запусков)
imoex_data = prepare_data(returns_imoex, None, None, 'IMOEX')
nasdaq_data = prepare_data(
    df_nasdaq['Close'].values, 
    df_nasdaq['Date'].values,
    {'train_end': '2018-01-01', 'val_start': '2018-01-01', 'val_end': '2020-01-01', 'test_start': '2020-01-01'},
    'NASDAQ'
)

# Объединение ДО нормализации
X_train_raw = np.concatenate([imoex_data['X_train_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                               nasdaq_data['X_train_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])
X_val_raw = np.concatenate([imoex_data['X_val_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                             nasdaq_data['X_val_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])
X_test_raw = np.concatenate([imoex_data['X_test_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                              nasdaq_data['X_test_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])

y_train_raw = np.concatenate([imoex_data['y_train_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                               nasdaq_data['y_train_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])
y_val_raw = np.concatenate([imoex_data['y_val_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                             nasdaq_data['y_val_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])
y_test_raw = np.concatenate([imoex_data['y_test_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                              nasdaq_data['y_test_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])

garch_train_raw = np.concatenate([imoex_data['garch_train_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                                   nasdaq_data['garch_train_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])
garch_val_raw = np.concatenate([imoex_data['garch_val_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                                 nasdaq_data['garch_val_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])
garch_test_raw = np.concatenate([imoex_data['garch_test_norm'] * imoex_data['train_std'] + imoex_data['train_mean'],
                                  nasdaq_data['garch_test_norm'] * nasdaq_data['train_std'] + nasdaq_data['train_mean']])

# ЕДИНАЯ нормализация
combined_mean = X_train_raw.mean()
combined_std = X_train_raw.std()

X_train_combined = (X_train_raw - combined_mean) / combined_std
X_val_combined = (X_val_raw - combined_mean) / combined_std
X_test_combined = (X_test_raw - combined_mean) / combined_std

y_train_combined = (y_train_raw - combined_mean) / combined_std
y_val_combined = (y_val_raw - combined_mean) / combined_std
y_test_combined = (y_test_raw - combined_mean) / combined_std

garch_train_combined = (garch_train_raw - combined_mean) / combined_std
garch_val_combined = (garch_val_raw - combined_mean) / combined_std
garch_test_combined = (garch_test_raw - combined_mean) / combined_std

# Метки рынков для теста
market_labels_test = np.array(['IMOEX'] * len(imoex_data['y_test_norm']) + ['NASDAQ'] * len(nasdaq_data['y_test_norm']))

# Сохраняем targets_real для каждого рынка отдельно (уже в исходном масштабе)
targets_imoex = imoex_data['targets_real']
targets_nasdaq = nasdaq_data['targets_real']

print(f'Combined mean: {combined_mean:.4f}, std: {combined_std:.4f}')
print(f'Combined train: {len(X_train_combined)}, val: {len(X_val_combined)}, test: {len(X_test_combined)}')

[IMOEX] Доходностей: 4076
[IMOEX] GT variance точек: 3982
[IMOEX] Окна: (3892, 90)
[IMOEX] Train: 2325, Val: 505, Test: 1062
[IMOEX] Считаю GARCH для всех окон...
  500/3892
  1000/3892
  1500/3892
  2000/3892
  2500/3892
  3000/3892
  3500/3892
[NASDAQ] Доходностей: 3913
[NASDAQ] GT variance точек: 3820
[NASDAQ] Окна: (3730, 90)
[NASDAQ] Train: 1928, Val: 529, Test: 1273
[NASDAQ] Считаю GARCH для всех окон...
  500/3730
  1000/3730
  1500/3730
  2000/3730
  2500/3730
  3000/3730
  3500/3730
Combined mean: 0.8078, std: 0.5520
Combined train: 4253, val: 1034, test: 2335


In [23]:
train_ld = DataLoader(VolatilityDataset(X_train_combined, y_train_combined), 128, shuffle=True)
val_ld = DataLoader(VolatilityDataset(X_val_combined, y_val_combined), 128)
test_ld = DataLoader(VolatilityDataset(X_test_combined, y_test_combined), 128)

ginn_train_ld = DataLoader(GatingDataset(X_train_combined, y_train_combined, garch_train_combined), 256, shuffle=True)
ginn_val_ld = DataLoader(GatingDataset(X_val_combined, y_val_combined, garch_val_combined), 128)
ginn_test_ld = DataLoader(GatingDataset(X_test_combined, y_test_combined, garch_test_combined), 128)

In [24]:
print('\n=== Обучение единой LSTM на двух рынках ===')
lstm_model = VolatilityLSTM(hidden_size=128).to(device)
optimizer = optim.AdamW(lstm_model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
criterion = nn.MSELoss()

best_val = float('inf')
patience_cnt, PATIENCE = 0, 50
best_state = None

for epoch in range(300):
    lstm_model.train()
    for X_b, y_b in train_ld:
        optimizer.zero_grad()
        loss = criterion(lstm_model(X_b.to(device)), y_b.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
    
    lstm_model.eval()
    val_losses = []
    with torch.no_grad():
        for X_b, y_b in val_ld:
            val_losses.append(criterion(lstm_model(X_b.to(device)), y_b.to(device)).item())
    
    avg_val = np.mean(val_losses)
    scheduler.step(avg_val)
    
    if avg_val < best_val:
        best_val = avg_val
        best_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
    
    if patience_cnt >= PATIENCE:
        print(f'LSTM early stop at epoch {epoch+1}')
        break
    if (epoch+1) % 20 == 0:
        print(f'LSTM Epoch {epoch+1}: Val Loss {avg_val:.4f}')

lstm_model.load_state_dict(best_state)


=== Обучение единой LSTM на двух рынках ===
LSTM Epoch 20: Val Loss 0.1469
LSTM Epoch 40: Val Loss 0.1988
LSTM Epoch 60: Val Loss 0.1443
LSTM Epoch 80: Val Loss 0.1509
LSTM Epoch 100: Val Loss 0.1184
LSTM Epoch 120: Val Loss 0.1321
LSTM Epoch 140: Val Loss 0.1051
LSTM Epoch 160: Val Loss 0.1093
LSTM Epoch 180: Val Loss 0.1386
LSTM Epoch 200: Val Loss 0.1058
LSTM early stop at epoch 209


<All keys matched successfully>

In [25]:
print('\n=== Обучение единой Feature Gating GINN на двух рынках ===')
ginn_model = FeatureGatingGINN(hidden_size=128).to(device)
optimizer = optim.AdamW(ginn_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
mse = nn.MSELoss()

best_val = float('inf')
patience_cnt, PATIENCE = 0, 50
best_state = None

for epoch in range(300):
    ginn_model.train()
    for X, y_r, y_g in ginn_train_ld:
        X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
        optimizer.zero_grad()
        pred, lam = ginn_model(X, y_g.unsqueeze(-1))
        loss = mse(pred, y_r) + 0.001 * ((lam - 0.5)**2).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ginn_model.parameters(), 1.0)
        optimizer.step()
    
    ginn_model.eval()
    val_loss = []
    with torch.no_grad():
        for X, y_r, y_g in ginn_val_ld:
            X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
            pred, lam = ginn_model(X, y_g.unsqueeze(-1))
            val_loss.append(mse(pred, y_r).item())
    
    avg_val = np.mean(val_loss)
    scheduler.step(avg_val)
    
    if avg_val < best_val:
        best_val = avg_val
        best_state = {k: v.cpu().clone() for k, v in ginn_model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
    
    if patience_cnt >= PATIENCE:
        print(f'Adaptive GINN early stop at epoch {epoch+1}')
        break
    if (epoch+1) % 20 == 0:
        print(f'Adaptive GINN Ep {epoch+1}: Val Loss {avg_val:.4f}')

ginn_model.load_state_dict(best_state)


=== Обучение единой Feature Gating GINN на двух рынках ===
Adaptive GINN Ep 20: Val Loss 0.1463
Adaptive GINN Ep 40: Val Loss 0.1233
Adaptive GINN Ep 60: Val Loss 0.1100
Adaptive GINN Ep 80: Val Loss 0.8279
Adaptive GINN Ep 100: Val Loss 0.1342
Adaptive GINN Ep 120: Val Loss 0.0962
Adaptive GINN Ep 140: Val Loss 0.0990
Adaptive GINN Ep 160: Val Loss 0.0983
Adaptive GINN early stop at epoch 167


<All keys matched successfully>

In [26]:
print('\n=== Тестирование ===')

# LSTM predictions
lstm_model.eval()
lstm_all_preds = []
with torch.no_grad():
    for X_b, y_b in test_ld:
        lstm_all_preds.extend(lstm_model(X_b.to(device)).cpu().numpy())
lstm_all_preds = np.array(lstm_all_preds)

# GINN predictions
ginn_model.eval()
ginn_all_preds, ginn_all_lambdas = [], []
with torch.no_grad():
    for X, y_r, y_g in ginn_test_ld:
        X, y_g = X.to(device), y_g.to(device)
        p, l = ginn_model(X, y_g.unsqueeze(-1))
        ginn_all_preds.extend(p.cpu().numpy())
        ginn_all_lambdas.extend(l.cpu().numpy())
ginn_all_preds = np.array(ginn_all_preds)
ginn_all_lambdas = np.array(ginn_all_lambdas)



=== Тестирование ===


In [27]:
# Денормализация с ЕДИНЫМИ параметрами
lstm_preds_denorm = np.expm1(lstm_all_preds * combined_std + combined_mean)
ginn_preds_denorm = np.expm1(ginn_all_preds * combined_std + combined_mean)

# Разделение по рынкам
imoex_mask = market_labels_test == 'IMOEX'
nasdaq_mask = market_labels_test == 'NASDAQ'

results = {}
for market_name, mask, targets, garch_raw in [
    ('IMOEX', imoex_mask, targets_imoex, imoex_data['garch_test_raw']),
    ('NASDAQ', nasdaq_mask, targets_nasdaq, nasdaq_data['garch_test_raw'])
]:
    # GARCH
    garch_mask = ~np.isnan(garch_raw)
    garch_mse = mean_squared_error(targets[garch_mask], garch_raw[garch_mask])
    garch_mae = mean_absolute_error(targets[garch_mask], garch_raw[garch_mask])
    garch_r2 = r2_score(targets[garch_mask], garch_raw[garch_mask])
    
    # LSTM
    lstm_mse = mean_squared_error(targets, lstm_preds_denorm[mask])
    lstm_mae = mean_absolute_error(targets, lstm_preds_denorm[mask])
    lstm_r2 = r2_score(targets, lstm_preds_denorm[mask])
    
    # GINN
    ginn_mse = mean_squared_error(targets, ginn_preds_denorm[mask])
    ginn_mae = mean_absolute_error(targets, ginn_preds_denorm[mask])
    ginn_r2 = r2_score(targets, ginn_preds_denorm[mask])
    
    results[market_name] = {
        'GARCH': {'MSE': garch_mse, 'MAE': garch_mae, 'R2': garch_r2},
        'LSTM': {'MSE': lstm_mse, 'MAE': lstm_mae, 'R2': lstm_r2},
        'GINN': {'MSE': ginn_mse, 'MAE': ginn_mae, 'R2': ginn_r2},
        'preds': {
            'LSTM': lstm_preds_denorm[mask],
            'GINN': ginn_preds_denorm[mask],
            'lambdas': ginn_all_lambdas[mask]
        }
    }

In [28]:
print('\n' + '-'*80)
print('РЕЗУЛЬТАТЫ ЕДИНОЙ МОДЕЛИ НА ДВУХ РЫНКАХ')
print('-'*80)
print(f"{'Рынок':<10} {'Модель':<12} {'MSE':>10} {'MAE':>10} {'R²':>10}")
print('-'*52)
for market in ['IMOEX', 'NASDAQ']:
    for model_name in ['GARCH', 'LSTM', 'GINN']:
        m = results[market][model_name]
        print(f"{market:<10} {model_name:<12} {m['MSE']:>10.2f} {m['MAE']:>10.4f} {m['R2']:>10.4f}")
print('-'*80)


--------------------------------------------------------------------------------
РЕЗУЛЬТАТЫ ЕДИНОЙ МОДЕЛИ НА ДВУХ РЫНКАХ
--------------------------------------------------------------------------------
Рынок      Модель              MSE        MAE         R²
----------------------------------------------------
IMOEX      GARCH            831.75     2.9592  -117.8625
IMOEX      LSTM              74.30     1.7341    -9.6185
IMOEX      GINN             431.77     2.5064   -60.7031
NASDAQ     GARCH             26.71     2.1934    -0.9501
NASDAQ     LSTM               7.88     2.0119     0.4248
NASDAQ     GINN              19.65     2.2108    -0.4350
--------------------------------------------------------------------------------


In [29]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('IMOEX: Прогнозы волатильности', 'NASDAQ: Прогнозы волатильности',
                    'IMOEX: Адаптивный λ(t)', 'NASDAQ: Адаптивный λ(t)'),
    vertical_spacing=0.12, horizontal_spacing=0.1
)

colors = {'GARCH': 'red', 'LSTM': 'orange', 'GINN': 'blue'}

for idx, (market, data) in enumerate([('IMOEX', imoex_data), ('NASDAQ', nasdaq_data)]):
    row = idx + 1
    
    # Прогнозы
    fig.add_trace(go.Scatter(x=data['dates_test'], y=data['targets_real'], 
                             name=f'{market} Real', line=dict(color='black', width=1), opacity=0.4), 
                  row=row, col=1)
    
    for model_name, color in colors.items():
        if model_name == 'GARCH':
            y_vals = data['garch_test_raw']
        else:
            y_vals = results[market]['preds'][model_name]
        fig.add_trace(go.Scatter(x=data['dates_test'], y=y_vals,
                                 name=f'{market} {model_name}', line=dict(color=color, width=1.5)), 
                      row=row, col=1)
    
    # Лямбда
    fig.add_trace(go.Scatter(x=data['dates_test'], y=results[market]['preds']['lambdas'],
                             name=f'{market} λ', line=dict(color='purple'), 
                             fill='tozeroy', fillcolor='rgba(128,0,128,0.1)'), 
                  row=row, col=2)
    fig.add_hline(y=0.5, line_dash='dash', line_color='green', row=row, col=2)

fig.update_layout(
    title='Единая Feature Gating GINN на двух рынках',
    height=900, showlegend=True,
    hovermode='x unified'
)
fig.show()

# Статистика по λ для каждого рынка
for market in ['IMOEX', 'NASDAQ']:
    lambs = results[market]['preds']['lambdas']
    print(f"\n{market} λ: mean={lambs.mean():.4f} ± {lambs.std():.4f}, "
          f"<0.3: {(lambs<0.3).mean()*100:.1f}%, >0.7: {(lambs>0.7).mean()*100:.1f}%")


IMOEX λ: mean=0.5697 ± 0.1745, <0.3: 3.3%, >0.7: 18.2%

NASDAQ λ: mean=0.5443 ± 0.1872, <0.3: 6.6%, >0.7: 18.5%
